<a href="https://colab.research.google.com/github/albormeha133/sustainspringhackathon/blob/main/Hackathon_Question_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install xgboost -q

import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Load files uploaded to Colab
Pesticides = pd.read_csv("pesticides.csv")
Yield = pd.read_csv("yield.csv")
Rainfall = pd.read_csv("rainfall.csv")
Temperature = pd.read_csv("temp.csv")

# Clean column names
Pesticides.columns = Pesticides.columns.str.strip()
Yield.columns = Yield.columns.str.strip()
Rainfall.columns = Rainfall.columns.str.strip()
Temperature.columns = Temperature.columns.str.strip()

# Standardise names
Temperature = Temperature.rename(columns={"country": "Area", "year": "Year"})
Rainfall = Rainfall.rename(columns={"average_rain_fall_mm_per_year": "Average_Rainfall"})
Yield = Yield.rename(columns={"Value": "Value_yield"})

# Merge all data
data = Yield.merge(
    Pesticides[["Area", "Year", "Value"]],
    on=["Area", "Year"],
    how="left"
).rename(columns={"Value": "Value_pest"})

data = data.merge(
    Rainfall[["Area", "Year", "Average_Rainfall"]],
    on=["Area", "Year"],
    how="left"
)

data = data.merge(
    Temperature[["Area", "Year", "avg_temp"]],
    on=["Area", "Year"],
    how="left"
)

data = data.dropna(subset=[
    "Value_yield", "Area", "Year", "Item",
    "Value_pest", "Average_Rainfall", "avg_temp"
])

country_x = "Canada"
split_year = 2008

train_data = data[data["Year"] <= split_year].copy()
test_data = data[data["Year"] > split_year].copy()

global_train = train_data.copy()
global_test = test_data[test_data["Area"] == country_x].copy()

local_train = train_data[train_data["Area"] == country_x].copy()
local_test = test_data[test_data["Area"] == country_x].copy()

print("Country:", country_x)
print("Global train rows:", len(global_train))
print("Global test rows:", len(global_test))
print("Local train rows:", len(local_train))
print("Local test rows:", len(local_test))

if len(global_test) == 0 or len(local_train) == 0 or len(local_test) == 0:
    raise ValueError(f"Not enough data for {country_x}")

feature_cols = ["Value_pest", "Year", "Item", "Area", "Average_Rainfall", "avg_temp"]

# Build encoded datasets once
X_global_train = pd.get_dummies(global_train[feature_cols])
X_global_test = pd.get_dummies(global_test[feature_cols])
X_global_test = X_global_test.reindex(columns=X_global_train.columns, fill_value=0)

X_local_train = pd.get_dummies(local_train[feature_cols])
X_local_test = pd.get_dummies(local_test[feature_cols])
X_local_test = X_local_test.reindex(columns=X_local_train.columns, fill_value=0)

y_global_train = global_train["Value_yield"]
y_global_test = global_test["Value_yield"]

y_local_train = local_train["Value_yield"]
y_local_test = local_test["Value_yield"]

results = []

def compare_model(model, model_name):
    global_model = model
    global_model.fit(X_global_train, y_global_train)
    global_preds = global_model.predict(X_global_test)
    global_mae = mean_absolute_error(y_global_test, global_preds)

    # create a fresh model of the same type for local training
    if model_name == "XGBoost":
        local_model = xgb.XGBRegressor(
            n_estimators=100,
            max_depth=4,
            random_state=42
        )
    else:
        local_model = RandomForestRegressor(
            n_estimators=100,
            random_state=42
        )

    local_model.fit(X_local_train, y_local_train)
    local_preds = local_model.predict(X_local_test)
    local_mae = mean_absolute_error(y_local_test, local_preds)

    results.append({
        "Model": model_name,
        "Country": country_x,
        "Global MAE": round(global_mae, 2),
        "Local MAE": round(local_mae, 2),
        "Winner": "Local model" if local_mae < global_mae else "Global model"
    })

compare_model(
    xgb.XGBRegressor(n_estimators=100, max_depth=4, random_state=42),
    "XGBoost"
)

compare_model(
    RandomForestRegressor(n_estimators=100, random_state=42),
    "Random Forest"
)

results_df = pd.DataFrame(results)
results_df

Country: Canada
Global train rows: 21997
Global test rows: 120
Local train rows: 432
Local test rows: 120


,Model,Country,Global MAE,Local MAE,Winner
0,XGBoost,Canada,15442.14,2839.11,Local model
1,Random Forest,Canada,3666.00,2874.57,Local model
